In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16 # pretrained VGG16 CNN model
from tensorflow.keras.models import Model # used to build custom model
from tensorflow.keras.layers import Dense, Flatten # layer used for classification
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator


#  ImageDataGenerator automatically loads images and performs preprocessing
datagen = ImageDataGenerator(

 # Normalize pixel values between 0 and 1
    rescale=1./255,

    # Reservation 20% of dataset for validation
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    'dataset',    # Dataset folder
    target_size=(224, 224), # Resize images to VGG16 size
    batch_size=32,   # number of images processed per iteration
    class_mode='categorical',  # Multiclass classification
    subset='training' # Specify this for training data

)

val_data = datagen.flow_from_directory(
    'dataset',    # Dataset folder
    target_size=(224, 224), # Resize images to VGG16 size
    batch_size=32,   # number of images processed per iteration
    class_mode='categorical',  # Multiclass classification
    subset='validation' # Specify this for validation data

)

base_model = VGG16(
    weights='imagenet',     # load wights trained on imagenet dataet
    include_top=False,    # Remove original classification layer
    input_shape=(224, 224, 3) # input image dimensions
)

for layer in base_model.layers:
    layer.trainable = False

# The following comments describe the model architecture conceptually
# input image
# conv blocks (VGG16)
# block5_pool - last layer of base model
# Flatten
# Dense
# output

# convert feature maps into 1D vector
x = Flatten()(base_model.output)

# add dense layer with 256 neurons
x = Dense(256, activation='relu')(x)

# output layer
# Number of neurons equals of persons
output = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, # input layer
              outputs=output
)

model.compile(
    optimizer=Adam(learning_rate=0.001), # gradient optimization
    loss='categorical_crossentropy', # loss nfunction
    metrics=['accuracy']    # measure model accuracy

)

model.fit(
    train_data,      # training dataset
    epochs=10,               # number of times model sees
    validation_data=val_data     # validation dataset
)

model.save('vgg16_face_recognition_model.h5')

